# Profilage des données - CyberGuard
Notebook d'Aya (Data Engineering). À exécuter **après** `prepare_data.py`. Il lit `data/processed_train.parquet` et `data/processed_test.parquet`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
train = pd.read_parquet(ROOT / "data" / "processed_train.parquet")
test = pd.read_parquet(ROOT / "data" / "processed_test.parquet")
print("train :", train.shape, "| test :", test.shape)
train.head()

## Colonnes et types

In [ ]:
print("Mêmes colonnes train/test :", list(train.columns) == list(test.columns))
train.dtypes.value_counts()

## Distribution des classes

In [ ]:
dist = pd.DataFrame({
    "train": train["label"].value_counts(),
    "test": test["label"].value_counts(),
}).fillna(0).astype(int)
dist["train_%"] = (100 * dist["train"] / dist["train"].sum()).round(3)
dist["test_%"] = (100 * dist["test"] / dist["test"].sum()).round(3)
dist

In [ ]:
ax = dist[["train", "test"]].plot(kind="bar", figsize=(11, 4), logy=True)
ax.set_title("Lignes par classe (échelle logarithmique)")
ax.set_ylabel("Nombre de lignes")
plt.tight_layout()
plt.show()

## Normal / attaque

In [ ]:
pd.DataFrame({
    "train": train["is_attack"].value_counts(normalize=True).round(4),
    "test": test["is_attack"].value_counts(normalize=True).round(4),
}).rename(index={0: "normal", 1: "attaque"})

## Qualité après nettoyage

In [ ]:
num = train.select_dtypes(include=[np.number])
print("Valeurs manquantes (train) :", int(train.isna().sum().sum()))
print("Valeurs infinies (train)   :", int(np.isinf(num).sum().sum()))
print("Doublons (train)           :", int(train.drop(columns=["record_id"]).duplicated().sum()))
only_test = set(test["label"]) - set(train["label"])
print("Classes présentes dans le test seulement :", only_test or "aucune")

## Conclusions
*À compléter par Aya : déséquilibre observé, classes trop rares pour des métriques fiables, colonnes supprimées, choix faits.*